# Creating the training/validation/test dataset USING ONLY **ROI IMAGES** AND **LABELS**

## CBIS-DDSM ROI Preprocessing and Patient-Level Splitting

This notebook prepares the CBIS-DDSM (Curated Breast Imaging Subset of DDSM) dataset for ROI classification (Calcification vs. Mass).

We will:

1. Load ROI images for calcification and mass cases.

2. Apply preprocessing (contrast enhancement, resizing, padding).

3. Perform patient-level separation to ensure no data leakage.

4. Combine all subsets for CNN training.

### Step 1 — Imports and Setup

In [1]:
import os
import cv2
import glob
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

### Step 2 — Define the Preprocessor Class

The class below:

- Loads and enhances grayscale ROI images.

- Resizes them to a fixed shape with padding to preserve aspect ratio.

- Splits the dataset at the **patient level**, ensuring that ROIs from the same patient are never shared between training and validation sets.

In [2]:
class CBIS_ROI_ClassifierPreprocessor:
    def __init__(self, img_size=(224, 224)):
        """
        Preprocessor for CBIS-DDSM ROI classification (calc vs mass).

        Args:
            img_size (tuple): Target image size (height, width).
        """
        self.img_size = img_size
        self.class_mapping = {'calc': 0, 'mass': 1}  # Binary labels

    def load_image(self, image_path):
        """Load a single image in grayscale."""
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not load image: {image_path}")
        return img

    def enhance_contrast(self, img):
        """Enhance contrast for mammogram ROIs using tuned CLAHE."""
        if img.dtype != np.uint8:
            img = (img * 255).astype(np.uint8)
    
        # Optional mild denoise
        img = cv2.medianBlur(img, 3)
    
        # Tuned CLAHE
        clahe = cv2.createCLAHE(clipLimit=1, tileGridSize=(8, 8))
        enhanced = clahe.apply(img)
    
        return enhanced


    def resize_with_padding(self, img):
        """
        Resize image while preserving aspect ratio.
        Pads the image to match the desired shape.
        """
        h, w = img.shape[:2]
        target_h, target_w = self.img_size
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        resized_img = cv2.resize(img, (new_w, new_h))
        padded_img = np.zeros((target_h, target_w), dtype=resized_img.dtype)
        x_offset = (target_w - new_w) // 2
        y_offset = (target_h - new_h) // 2
        padded_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_img
        return padded_img

    def process_image(self, image_path, contrast):
        """Apply all preprocessing steps to one image."""
        img = self.load_image(image_path)
        if contrast:
            img = self.enhance_contrast(img)
        img = self.resize_with_padding(img)
        return img






    def load_dataset_patient_level_unified(self, data_folders, label_name, train_ratio=0.7, val_ratio=0.15, seed=42, debug=False):
        """
        Load a dataset with a strict 70/15/15 (train/val/test) split at patient level,
        combining patients from multiple source folders.
    
        Args:
            data_folders (list[str]): List of folders containing patient subfolders.
            label_name (str): 'calc' or 'mass'.
            train_ratio (float): Fraction of patients for training.
            val_ratio (float): Fraction for validation (rest goes to test).
            seed (int): Random seed for reproducibility.
            debug (bool): If True, prints detailed patient counts and overlap checks.
    
        Returns:
            X_train, y_train, X_val, y_val, X_test, y_test
        """
        # --- Collect all patient IDs across all folders ---
        patient_to_folder = {}
        for folder in data_folders:
            for d in os.listdir(folder):
                patient_dir = os.path.join(folder, d)
                if os.path.isdir(patient_dir):
                    if d in patient_to_folder:
                        print(f" Warning: duplicate patient ID {d} found in multiple folders!")
                    patient_to_folder[d] = folder
    
        all_patients = list(patient_to_folder.keys())
        np.random.seed(seed) 
        np.random.shuffle(all_patients)
    
        # --- Compute split sizes ---
        n_total = len(all_patients)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        n_test = n_total - n_train - n_val
    
        train_patients = all_patients[:n_train]
        val_patients   = all_patients[n_train:n_train + n_val]
        test_patients  = all_patients[n_train + n_val:]

        def load_patients(patient_list, contrast):
            X, y = [], []
            for pid in tqdm(patient_list, desc=f"Processing {label_name} patients"):
                folder = patient_to_folder[pid]
                patient_dir = os.path.join(folder, pid)
                image_paths = glob.glob(os.path.join(patient_dir, "*.png"))
                for image_path in image_paths:
                    try:
                        img = self.process_image(image_path, contrast)
                        X.append(img)
                        y.append(self.class_mapping[label_name])
                    except Exception as e:
                        print(f"Error processing {image_path}: {e}")
            X = np.expand_dims(np.array(X), axis=-1)
            y = np.array(y)
            return X, y
    
        # --- Load each split ---
        X_train_non_clahe, y_train_non_clahe = load_patients(train_patients, False)
        X_train_clahe, y_train_clahe = load_patients(train_patients, True)

        X_val_non_clahe, y_val_non_clahe = load_patients(val_patients, False)
        
        X_test, y_test   = load_patients(test_patients, False)
    
        if debug:
            print(f"\n{label_name}:")
            print(f"  Train patients: {len(train_patients)}")
            print(f"  Val patients:   {len(val_patients)}")
            print(f"  Test patients:  {len(test_patients)}")
            print(f"  Total patients: {n_total}")
    
            # Overlap checks
            sets = [set(train_patients), set(val_patients), set(test_patients)]
            print(f"  Overlaps:",
                  f"train-val={len(sets[0] & sets[1])},",
                  f"train-test={len(sets[0] & sets[2])},",
                  f"val-test={len(sets[1] & sets[2])}")
    
            print(f"  Image counts -> Train: {len(X_train_clahe)}, Val: {len(X_val_non_clahe)}, Test: {len(X_test)}")
    
        return X_train_non_clahe, y_train_non_clahe, X_val_non_clahe, y_val_non_clahe, X_train_clahe, y_train_clahe, X_test, y_test



### Step 3 — Load Dataset with Patient-Level Splitting

We’ll now define a method that:

- Loads all patient folders.

- Splits them into **train / validation / test** sets based on patient IDs.

- Loads all ROI .png files for each patient.

- Applies preprocessing.

### Step 4 — Define Dataset Paths


Now we specify the paths to each subset.
Each folder should contain patient subfolders (e.g. P_1234/roi_1.png, P_1234/roi_2.png, etc.)

In [3]:
base_dir = r"D:\cbis-ddsm_dataset_licenta\data\processed_separate_roi_full_masks"
preprocessor = CBIS_ROI_ClassifierPreprocessor(img_size=(224, 224))

# Directories for each class
calc_train_dir = os.path.join(base_dir, "calc_case_description_train_set_png", "roi_crops")
calc_test_dir  = os.path.join(base_dir, "calc_case_description_test_set_png", "roi_crops")
mass_train_dir = os.path.join(base_dir, "mass_case_description_train_set_png", "roi_crops")
mass_test_dir  = os.path.join(base_dir, "mass_case_description_test_set_png", "roi_crops")


### Step 5 — Load and Preprocess Data
We now load both calcification and mass datasets, applying the same preprocessing pipeline.

In [4]:
X_train_non_clahe_calc, y_train_non_clahe_calc, X_val_non_clahe_calc, y_val_non_clahe_calc, X_train_clahe_calc, y_train_clahe_calc, X_test_calc, y_test_calc = preprocessor.load_dataset_patient_level_unified(
    data_folders=[
        os.path.join(base_dir, "calc_case_description_train_set_png", "roi_crops"),
        os.path.join(base_dir, "calc_case_description_test_set_png", "roi_crops")
    ],
    label_name="calc",
    train_ratio=0.7,
    val_ratio=0.15,
    seed=13,
    debug=True
)

X_train_non_clahe_mass, y_train_non_clahe_mass, X_val_non_clahe_mass, y_val_non_clahe_mass, X_train_clahe_mass, y_train_clahe_mass, X_test_mass, y_test_mass = preprocessor.load_dataset_patient_level_unified(
    data_folders=[
        os.path.join(base_dir, "mass_case_description_train_set_png", "roi_crops"),
        os.path.join(base_dir, "mass_case_description_test_set_png", "roi_crops")
    ],
    label_name="mass",
    train_ratio=0.7,
    val_ratio=0.15,
    seed=13,
    debug=True
)

Processing calc patients: 100%|█████████████████████████████████████████████████████| 114/114 [00:00<00:00, 116.15it/s]



calc:
  Train patients: 527
  Val patients:   112
  Test patients:  114
  Total patients: 753
  Overlaps: train-val=0, train-test=0, val-test=0
  Image counts -> Train: 1318, Val: 256, Test: 298


Processing mass patients: 100%|█████████████████████████████████████████████████████| 135/135 [00:00<00:00, 171.28it/s]


mass:
  Train patients: 624
  Val patients:   133
  Test patients:  135
  Total patients: 892
  Overlaps: train-val=0, train-test=0, val-test=0
  Image counts -> Train: 1170, Val: 257, Test: 269
